In [6]:
from dataclasses import dataclass

@dataclass
class Instruction:
    name: str
    destination: str
    sources: list

# -------------------------------------------------
# 1. Fungsi Deteksi Dependency (RAW, WAR, WAW)
# -------------------------------------------------

def check_raw(inst1, inst2):
    """RAW (Read-After-Write): inst2 membaca register yang ditulis inst1"""
    return inst1.destination in inst2.sources

def check_war(inst1, inst2):
    """WAR (Write-After-Read): inst2 menulis register yang dibaca inst1"""
    return any(src in inst1.sources for src in [inst2.destination])

def check_waw(inst1, inst2):
    """WAW (Write-After-Write): inst2 menulis register yang sama dengan inst1"""
    return inst1.destination == inst2.destination and inst1.destination != ""

def show_dependencies(instructions):
    print("\n" + "=" * 50)
    print("ANALISIS DEPENDENCY (RAW, WAR, WAW)")
    print("=" * 50)

    found = False
    for i in range(len(instructions)):
        for j in range(i + 1, len(instructions)):
            inst1 = instructions[i]
            inst2 = instructions[j]

            if check_raw(inst1, inst2):
                print(f"[RAW Hazard]  : {inst2.name} membaca {inst1.destination} yang ditulis oleh {inst1.name}")
                found = True

            if check_war(inst1, inst2):
                print(f"[WAR Hazard]  : {inst2.name} menulis {inst2.destination} yang dibaca oleh {inst1.name}")
                found = True

            if check_waw(inst1, inst2):
                print(f"[WAW Hazard]  : {inst2.name} menulis {inst2.destination} yang juga ditulis oleh {inst1.name}")
                found = True

    if not found:
        print("Tidak ditemukan dependency/hazard.")

# -------------------------------------------------
# 2. Sequential Execution
# -------------------------------------------------

def sequential_execution(instructions):
    print("\n" + "=" * 50)
    print("SEQUENTIAL EXECUTION")
    print("=" * 50)

    cycle = 0
    for instruction in instructions:
        cycle += 1
        print(f"Cycle {cycle}: Executing {instruction.name} ({instruction.destination} = {' + '.join(instruction.sources)})")

    print(f"\nTotal Sequential Cycles: {cycle}")
    return cycle

# -------------------------------------------------
# 3. ILP / Parallel Execution
# -------------------------------------------------

def parallel_execution(instructions):
    print("\n" + "=" * 50)
    print("INSTRUCTION-LEVEL PARALLEL EXECUTION")
    print("=" * 50)

    completed = set()
    remaining = instructions.copy()
    cycle = 0

    while remaining:
        cycle += 1
        current_cycle = []

        for instruction in remaining:
            dependencies_satisfied = True
            for previous in instructions:
                if previous.name == instruction.name:
                    break
                # Cek jika ada RAW hazard yang belum selesai
                if check_raw(previous, instruction) and previous.name not in completed:
                    dependencies_satisfied = False
                    break

            if dependencies_satisfied:
                current_cycle.append(instruction)

        print(f"\nCycle {cycle}:")
        for instruction in current_cycle:
            print(f"  -> {instruction.name}: {instruction.destination} = {' + '.join(instruction.sources)}")

        for instruction in current_cycle:
            completed.add(instruction.name)
            remaining.remove(instruction)

    print(f"\nTotal ILP Cycles: {cycle}")
    return cycle

# -------------------------------------------------
# 4. Set Instruksi Pengujian (Mencakup RAW, WAR, WAW)
# -------------------------------------------------

instructions = [
    Instruction("I1", "R1", ["R2", "R3"]),  # I1 menulis R1, membaca R2 dan R3
    Instruction("I2", "R4", ["R1", "R5"]),  # I2 membaca R1 -> RAW dengan I1
    Instruction("I3", "R2", ["R6", "R7"]),  # I3 menulis R2 -> WAR dengan I1 (I1 membaca R2)
    Instruction("I4", "R1", ["R8", "R9"]),  # I4 menulis R1 -> WAW dengan I1 (I1 menulis R1)
]

# -------------------------------------------------
# 5. Eksekusi Program Utama
# -------------------------------------------------

def main():
    print("\n" + "*" * 60)
    print("      INSTRUCTION-LEVEL PARALLELISM SIMULATOR")
    print("*" * 60)

    print("\nInstruction Set:")
    for instruction in instructions:
        print(f"{instruction.name}: {instruction.destination} = {' + '.join(instruction.sources)}")

    # Deteksi RAW, WAR, WAW
    show_dependencies(instructions)

    # Simulasi Sekuensial dan Paralel
    seq_cycles = sequential_execution(instructions)
    par_cycles = parallel_execution(instructions)

    # Kalkulasi Performa
    speedup = seq_cycles / par_cycles
    efficiency = (speedup / len(instructions)) * 100

    print("\n" + "=" * 50)
    print("PERFORMANCE COMPARISON")
    print("=" * 50)
    print(f"Sequential Cycles : {seq_cycles}")
    print(f"ILP Cycles        : {par_cycles}")
    print(f"Speedup           : {speedup:.2f}x")
    print(f"ILP Efficiency    : {efficiency:.2f}%")

if __name__ == "__main__":
    main()


************************************************************
      INSTRUCTION-LEVEL PARALLELISM SIMULATOR
************************************************************

Instruction Set:
I1: R1 = R2 + R3
I2: R4 = R1 + R5
I3: R2 = R6 + R7
I4: R1 = R8 + R9

ANALISIS DEPENDENCY (RAW, WAR, WAW)
[RAW Hazard]  : I2 membaca R1 yang ditulis oleh I1
[WAR Hazard]  : I3 menulis R2 yang dibaca oleh I1
[WAW Hazard]  : I4 menulis R1 yang juga ditulis oleh I1
[WAR Hazard]  : I4 menulis R1 yang dibaca oleh I2

SEQUENTIAL EXECUTION
Cycle 1: Executing I1 (R1 = R2 + R3)
Cycle 2: Executing I2 (R4 = R1 + R5)
Cycle 3: Executing I3 (R2 = R6 + R7)
Cycle 4: Executing I4 (R1 = R8 + R9)

Total Sequential Cycles: 4

INSTRUCTION-LEVEL PARALLEL EXECUTION

Cycle 1:
  -> I1: R1 = R2 + R3
  -> I3: R2 = R6 + R7
  -> I4: R1 = R8 + R9

Cycle 2:
  -> I2: R4 = R1 + R5

Total ILP Cycles: 2

PERFORMANCE COMPARISON
Sequential Cycles : 4
ILP Cycles        : 2
Speedup           : 2.00x
ILP Efficiency    : 50.00%
